# 04 · Asal grup, pemilihan 109 label, dan fold

**File:** `04_Grup_Split_dan_Fold.ipynb`

Di sini kita mencari tahu **kenapa molekul tertentu masuk train atau test**, dan bagaimana train dibagi lagi untuk validasi. *Train* dipakai belajar model; *test* dipakai evaluasi akhir; *fold* adalah bagian data yang bergantian dipakai untuk validasi.

Kita menelusuri pembentukan **kohor**, yaitu kumpulan molekul yang dipakai penelitian, lalu mencocokkannya dengan pembagian yang sudah disimpan. Run All pada notebook ini hanya memeriksa data di memori; tidak melatih model, mengunduh data, atau menimpa hasil.

Urutan belajar: **03_Dataset_dan_Fitur.ipynb → notebook ini → 05_Model_dan_Bobot.ipynb**. Setiap notebook memuat datanya sendiri, jadi tidak bergantung pada variabel notebook sebelumnya.

Kode yang ditampilkan berasal dari fungsi aktif di `KODE_ML/03_Kode/`, serta fungsi audit sejarah di `asal_kohor.py`. Menampilkan kode fungsi belum berarti menjalankan isi fungsi itu. Data dan model besar tetap dibaca dari folder artefak proyek.

**Prioritas membaca (revisi 6 November 2026)**

**Inti:** grup dari struktur tanpa stereo atau Morgan historis 1.024 bit identik, split/fold per label, seed pembagian versus seed model, dan batas kemiripan yang belum dipisahkan. **Pendalaman:** AST, union-find, fallback, serta rekonstruksi seluruh indeks. Aturan grup dan hasil cek tetap inti; rincian implementasinya dapat dibaca belakangan.


## Persiapan [Rujukan fungsi]

Sel ini mencari folder `KODE_ML` dan memuat kode lokal. Perhatikan alamat **Kode aktif** pada output: alamat itu menunjukkan salinan proyek yang sedang dipakai.


In [1]:
from pathlib import Path
import sys
sys.dont_write_bytecode = True
cari = [Path.cwd(), *Path.cwd().parents]
KODE = next((p if (p / "03_Kode/runtime.py").is_file() and p.name == "KODE_ML"
             else p / "KODE_ML" for p in cari
             if (p / "03_Kode/runtime.py").is_file() and p.name == "KODE_ML"
             or (p / "KODE_ML/03_Kode/runtime.py").is_file()), None)
if KODE is None:
    raise FileNotFoundError("Buka notebook dari folder proyek atau KODE_ML.")
sys.path.insert(0, str(KODE.parent))
sys.path.insert(0, str(KODE))
from research.runtime import ROOT, read, digest
from belajar import tampilkan_sumber, tampilkan_awal
import pandas as pd
import numpy as np
from IPython.display import display, Markdown
pd.set_option("display.max_colwidth", 85)
print("Kode aktif:", KODE)


Kode aktif: C:\Users\ACER\Documents\Marvel\Skripsi\Project Skripsi\Perfume-MultiLabel-Classifier\KODE_ML


## Bedakan pembentukan historis dengan pemakaian saat ini [Inti]

`asal_kohor.py` membantu menelusuri **bagaimana data dan grup dulu dibentuk**. Tiga fungsi aslinya mempertahankan logika kode yang sama, yang diperiksa lewat struktur kode atau AST.

Fungsi ini dipakai untuk audit notebook, bukan oleh program training utama saat ini. Pembagian yang dibentuk ulang hanya disimpan di memori untuk dicocokkan; tidak mengganti pembagian penelitian.


In [2]:
import asal_kohor as asal
from research import dataset, splits
from research.preprocessing import load_config, taxonomy_from_ifra, load_sources
tampilkan_awal(asal)

**Import dan konstanta aktual:** `KODE_ML/asal_kohor.py`. SHA-256: `c0b4fab50851629f320eacb5c41b6d41b592d9fecd703dde1da13788e0c066e5`.

```python
"""Audit asal kohor beku. Bukan pembangun dataset baru atau fitur model 1024.

Tiga fungsi historis dipertahankan persis secara AST dari source pembentuk kohor.
Berikan fingerprint 1024 tersimpan untuk merekonstruksi identitas grup historis.
Fitur model utama tetap dibuat oleh research.features: Morgan 2048 + 8.
"""
import numpy as np
import pandas as pd
from rdkit import Chem
from sklearn.model_selection import GroupShuffleSplit
from research.splits import POLICY, inner_folds, validate_grouped_split

```

## structure_feature_groups [Inti]

**Cara baca:** aturan penggabungan dan hubungan berantai adalah inti. Tabel `parent`/`find`/`union` berikut merupakan pendalaman implementasi; tidak perlu dihafal.

Fungsi ini memasukkan molekul ke **grup yang sama** jika struktur tanpa stereokimianya identik **atau** fingerprint historisnya identik. Hubungannya bisa berantai: jika A terhubung dengan B, lalu B dengan C, ketiganya masuk satu grup.

| Bagian kode | Cara membacanya |
|---|---|
| `parent` | Awalnya setiap baris dianggap punya grup sendiri. |
| `find` | Mencari wakil suatu grup; hasil pencarian dirapikan supaya pencarian berikutnya lebih cepat. |
| `union` | Menyatukan dua grup. Wakil berindeks lebih kecil dipilih agar hasilnya konsisten. |
| `structures` | Mencocokkan SMILES kanonik setelah informasi stereo dihapus. Isotop dan muatan tetap diperhatikan. |
| `fingerprints` | Mencocokkan fingerprint yang isinya sama persis. |
| `roots` dan `registry` | Memberi nomor grup akhir tanpa membuang baris molekul. |

Ini **bukan** pengelompokan semua molekul yang sekadar mirip. Kode tidak memakai batas kemiripan Tanimoto.


In [3]:
tampilkan_sumber(asal.structure_feature_groups)

**Source aktual:** `KODE_ML/asal_kohor.py`, baris 16–48. Kode di bawah dibaca dari objek yang diimpor. SHA-256: `c0b4fab50851629f320eacb5c41b6d41b592d9fecd703dde1da13788e0c066e5`.

```python
def structure_feature_groups(smiles, morgan):
    if morgan.ndim != 2 or len(smiles) != len(morgan):
        raise ValueError("Grouping input shape mismatch.")
    parent = list(range(len(smiles)))

    def find(i):
        while parent[i] != i:
            parent[i] = parent[parent[i]]
            i = parent[i]
        return i

    def union(a, b):
        a, b = find(a), find(b)
        parent[max(a, b)] = min(a, b)

    structures, fingerprints = {}, {}
    for i, s in enumerate(smiles):
        mol = Chem.MolFromSmiles(s)
        if mol is None or len(Chem.GetMolFrags(mol)) != 1:
            raise ValueError("Grouping requires valid single molecules.")
        Chem.RemoveStereochemistry(mol)
        # Preserve isotope/charge; only remove stereo information for the grouping key.
        key = Chem.MolToSmiles(mol, isomericSmiles=True)
        fp = np.ascontiguousarray(morgan[i]).tobytes()
        if key in structures:
            union(i, structures[key])
        if fp in fingerprints:
            union(i, fingerprints[fp])
        structures.setdefault(key, i)
        fingerprints.setdefault(fp, i)
    roots = [find(i) for i in range(len(smiles))]
    registry = {root: j for j, root in enumerate(sorted(set(roots)))}
    return np.asarray([registry[r] for r in roots], dtype=np.int64)

```

## outer_split [Inti]

`GroupShuffleSplit` memilih sekitar **20% grup** untuk test dengan seed 42. Seed adalah pengaturan pengacakan agar pembagian dapat diulangi.

Yang dibagi adalah grup. Karena jumlah molekul tiap grup berbeda, jumlah molekul test tidak harus tepat 20%.


In [4]:
tampilkan_sumber(asal.outer_split)

**Source aktual:** `KODE_ML/asal_kohor.py`, baris 51–55. Kode di bawah dibaca dari objek yang diimpor. SHA-256: `c0b4fab50851629f320eacb5c41b6d41b592d9fecd703dde1da13788e0c066e5`.

```python
def outer_split(groups, test_size, seed):
    # Test allocation uses structure groups only, never label scores or test performance.
    cv = GroupShuffleSplit(n_splits=1, test_size=test_size, random_state=seed)
    a, b = next(cv.split(np.zeros(len(groups)), groups=groups))
    return np.sort(a), np.sort(b)

```

## encode_and_split [Inti]

Bagian ini mengubah daftar anotasi menjadi matriks target `y`, lalu menentukan label mana yang cukup didukung data train.

Urutannya penting:
1. Buat kolom target untuk seluruh istilah dalam kamus IFRA; belum memilih 109 label.
2. Bentuk grup, lalu pisahkan train dan test.
3. Hitung jumlah positif dan negatif **hanya pada train**. Label dipilih jika punya **≥30 positif dan ≥2 negatif**.
4. Ambil kolom label terpilih dengan urutan tetap. Baris molekul tetap dipertahankan.
5. Buat fold validasi untuk setiap label, lalu periksa pembagiannya.

`support` dan `selection` mencatat jumlah contoh serta alasan label dipilih atau tidak. Angka **109 adalah hasil aturan ini**, bukan mengambil 109 label teratas atau memilih label dari skor test. Hasilnya dikembalikan ke memori untuk dibandingkan dengan data tersimpan.


In [5]:
tampilkan_sumber(asal.encode_and_split)

**Source aktual:** `KODE_ML/asal_kohor.py`, baris 58–89. Kode di bawah dibaca dari objek yang diimpor. SHA-256: `c0b4fab50851629f320eacb5c41b6d41b592d9fecd703dde1da13788e0c066e5`.

```python
def encode_and_split(records, taxonomy, config, *, morgan=None):
    y = np.asarray([[int(label in row) for label in taxonomy] for row in records.labels], dtype=np.uint8)
    if morgan is None:
        morgan, _ = extract_features(records, config)
    groups = structure_feature_groups(records.smiles.tolist(), morgan)
    train, test = outer_split(groups, config["split"]["test_size"], config["split"]["seed"])
    positive = y[train].sum(axis=0)
    minimum = config["labels"]["min_train_positive"]
    chosen = [j for j in range(len(taxonomy)) if positive[j] >= minimum and len(train)-positive[j] >= 2]
    if not chosen:
        raise ValueError("No labels have enough training examples.")
    labels, target = [taxonomy[j] for j in chosen], y[:, chosen]
    folds, support = {}, []
    for j, label in enumerate(labels):
        pos = int(target[train,j].sum()); neg = len(train)-pos
        try:
            folds[label] = inner_folds(train, target[:,j], groups,
                                       config["split"]["max_folds"], config["split"]["seed"])
        except ValueError as exc:
            raise ValueError(f"Label {label}: {exc}") from exc
        support.append({"label":label,"train_positive":pos,"train_negative":neg,
                        "folds":len(folds[label]),
                        "positive_groups":len(set(groups[train][target[train,j] == 1])),
                        "negative_groups":len(set(groups[train][target[train,j] == 0]))})
    split = {"train":train.tolist(),"test":test.tolist(),"folds":folds,"eligible_labels":labels,
             "seed":config["split"]["seed"],"group_policy":POLICY,"groups":groups.tolist(),
             "outer_method":"GroupShuffleSplit","inner_method":"StratifiedGroupKFold",
             "test_size_unit":"groups"}
    validate_grouped_split(split, groups)
    selection = [{"label":label,"train_positive":int(positive[j]),"selected":j in chosen}
                 for j,label in enumerate(taxonomy)]
    return target, labels, split, pd.DataFrame(support), pd.DataFrame(selection)

```

## inner_folds [Inti]

Untuk satu label, `StratifiedGroupKFold` membagi train menjadi beberapa fold sambil menjaga grup tetap utuh dan mempertimbangkan kelas 0/1.

Satu giliran menahan satu fold untuk validasi, sementara bagian lainnya dipakai fitting. Batasnya lima fold; jumlahnya dapat dikurangi jika kelasnya tidak memungkinkan, tanpa melihat skor model. Pada data penelitian ini seluruh label mendapat lima fold. **Pembagian fold bisa berbeda antarlabel.**


In [6]:
tampilkan_sumber(splits.inner_folds)

**Source aktual:** `KODE_ML/03_Kode/splits.py`, baris 6–18. Kode di bawah dibaca dari objek yang diimpor. SHA-256: `9386cb3804ad90592c9a3b9e11cbffcb28023754ab069e10fc82c4aa1e90d2f0`.

```python
def inner_folds(train, target, groups, max_folds, seed):
    train = np.asarray(train, dtype=int)
    truth, train_groups = target[train], groups[train]
    if set(np.unique(truth)) != {0, 1}:
        raise ValueError("Grouped CV requires both target classes.")
    upper = min(max_folds, len(set(train_groups[truth == 1])), len(set(train_groups[truth == 0])))
    # Lower k only for class feasibility, never based on model scores. No random-split fallback.
    for k in range(upper, 1, -1):
        cv = StratifiedGroupKFold(n_splits=k, shuffle=True, random_state=seed)
        candidates = list(cv.split(np.zeros(len(train)), truth, train_groups))
        if all(len(np.unique(truth[a])) == 2 and len(np.unique(truth[b])) == 2 for a,b in candidates):
            return [{"train":train[a].tolist(), "validation":train[b].tolist()} for a,b in candidates]
    raise ValueError("No feasible grouped CV with both classes; keep molecules and review the protocol.")

```

## validate_grouped_split [Rujukan fungsi]

Ini pemeriksaan pembagian data: apakah ada indeks ganda, irisan train-test, atau anggota satu grup yang berada di dua sisi pembagian.

Kode juga memastikan setiap baris train mendapat giliran sebagai validation **tepat satu kali untuk setiap label** dalam satu putaran CV.


In [7]:
tampilkan_sumber(splits.validate_grouped_split)

**Source aktual:** `KODE_ML/03_Kode/splits.py`, baris 21–52. Kode di bawah dibaca dari objek yang diimpor. SHA-256: `9386cb3804ad90592c9a3b9e11cbffcb28023754ab069e10fc82c4aa1e90d2f0`.

```python
def validate_grouped_split(split, groups):
    groups = np.asarray(groups)
    if split.get("group_policy") != POLICY or split.get("groups") != groups.tolist():
        raise ValueError("Missing or inconsistent structure/feature groups.")
    universe = set(range(len(groups)))
    def partition(a, b, expected):
        if len(a) != len(set(a)) or len(b) != len(set(b)):
            raise ValueError("Duplicate split indices.")
        a, b = set(a), set(b)
        if not a or not b or a & b or a | b != expected:
            raise ValueError("Invalid grouped partition coverage.")
        if set(groups[list(a)]) & set(groups[list(b)]):
            raise ValueError("Structure/feature group leakage across partitions.")
    partition(split["train"], split["test"], universe)
    training = set(split["train"])
    if set(split["folds"]) != set(split["eligible_labels"]):
        raise ValueError("Fold label registry mismatch.")
    fold_count = 0
    for label, folds in split["folds"].items():
        if len(folds) < 2:
            raise ValueError("At least two grouped folds required.")
        seen = []
        for fold in folds:
            partition(fold["train"], fold["validation"], training)
            seen.extend(fold["validation"])
            fold_count += 1
        if sorted(seen) != sorted(training):
            raise ValueError("Grouped validation must cover training rows once.")
    counts = pd.Series(groups).value_counts()
    return {"group_policy":POLICY, "group_count":int(len(counts)),
            "largest_group":int(counts.max()), "group_overlap_train_test":0,
            "group_overlap_cv":0, "folds_verified":fold_count}

```

## Perbedaan teknis pada fallback historis [Pendalaman]

Bagian ini adalah detail audit; kamu bisa membacanya setelah alur grup dan pembagian data sudah paham.

Di `KODE_ML/asal_kohor.py`, fungsi berikut sengaja menghentikan proses jika audit mencoba membuat fitur lama sendiri. Karena itu, notebook memberikan fingerprint historis yang sudah tersimpan kepada `encode_and_split`.

```python
def extract_features(records, config):
    raise ValueError("Audit asal kohor wajib menerima morgan historis tersimpan; tidak membuat fitur lama.")
```

Pembatasan ini menjaga audit memakai masukan historis yang tepat. Logika `encode_and_split`, aturan grup, dan fitur model utama tidak diubah.


## Rekonstruksi pembentukan kohor di memori [Pendalaman]

Sel berikut membaca **Morgan 1024 bit yang sudah tersimpan** untuk memeriksa asal grup dan pembagian lama. Fitur model utama tetap **Morgan 2048 bit**, dengan delapan deskriptor tambahan untuk B/D.

`asal.encode_and_split` menerima fingerprint historis itu secara langsung. Perhatikan pemeriksaan `assert`: hasil grup, indeks train-test, fold, urutan label, dan target harus cocok dengan file penelitian. Tidak ada fitur 1024 baru atau pembagian baru yang ditulis ke file.


In [8]:
config = load_config()
tables,_ = load_sources(config)
taxonomy = taxonomy_from_ifra(tables['ifra_behavior'])
records = pd.read_csv(dataset.DATASET/'records.csv')
import json
for column in ('labels','sources'):
    records[column] = records[column].map(json.loads)
stored = read(dataset.DATASET/'splits.json')
historical_morgan = np.empty((len(records),1024),dtype=np.uint8)
for role in ('train','test'):
    with np.load(dataset.DATASET/f'{role}.npz',allow_pickle=False) as z:
        historical_morgan[z['row_index']] = z['morgan']
target, labels, rebuilt, supports, selection = asal.encode_and_split(records,taxonomy,config,morgan=historical_morgan)
assert rebuilt == stored
assert labels == read(dataset.DATASET/'dataset_manifest.json')['labels']
for role in ('train','test'):
    with np.load(dataset.DATASET/f'{role}.npz',allow_pickle=False) as z:
        assert np.array_equal(target[rebuilt[role]],z['truth'])
display({'taksonomi':len(taxonomy),'label_terpilih':len(labels),
 'train':len(rebuilt['train']),'test':len(rebuilt['test']),
 'split_grup_fold_target_identik':True})
display(splits.validate_grouped_split(rebuilt,np.asarray(rebuilt['groups'])))
display(selection.groupby('selected').agg(jumlah_label=('label','size'),min_positif=('train_positive','min'),max_positif=('train_positive','max')))

{'taksonomi': 184,
 'label_terpilih': 109,
 'train': 5353,
 'test': 1333,
 'split_grup_fold_target_identik': True}

{'group_policy': 'structure-or-morgan-components-v1',
 'group_count': 5363,
 'largest_group': 38,
 'group_overlap_train_test': 0,
 'group_overlap_cv': 0,
 'folds_verified': 545}

,jumlah_label,min_positif,max_positif
selected,,,
False,75,1,28
True,109,32,1987


## Contoh pembagian satu label [Inti]

Kita ambil **satu label** dan lihat jumlah baris fitting serta validasinya pada setiap fold. Kolom `grup_overlap` harus nol: tidak ada grup yang muncul di kedua bagian pada giliran yang sama.

Lima fold ini berasal dari pembagian data train, **bukan** dari lima sumber dataset. Antar-kandidat, fold untuk label dan seed yang sama dipakai bersama supaya pembandingannya konsisten.


In [9]:
label=labels[0]
groups=np.asarray(rebuilt['groups'])
display(pd.DataFrame([{'label':label,'fold':j+1,'latih':len(f['train']),
 'validasi':len(f['validation']),'grup_overlap':len(set(groups[f['train']]) & set(groups[f['validation']]))}
 for j,f in enumerate(rebuilt['folds'][label])]))
protocol=read(KODE/'03_Kode/config/experiment.json')
print('Seed repeated CV:',protocol['robustness']['seeds'])

,label,fold,latih,validasi,grup_overlap
0,acidic,1,4266,1087,0
1,acidic,2,4302,1051,0
2,acidic,3,4258,1095,0
3,acidic,4,4296,1057,0
4,acidic,5,4290,1063,0


Seed repeated CV: [7, 21, 42, 84, 100]


Pada validasi berulang, seed **7, 21, 42, 84, dan 100** masing-masing menghasilkan satu putaran pembagian fold. Angka 7 adalah pengaturan pengacakan, bukan perintah mengulang tujuh kali.

Seed pembagian ini berbeda dari `random_state` model, yang tetap **42** di `make_model`. Lima seed × lima fold berarti **25 proses fitting-validasi per label per kandidat** pada tahap ini; hitungan Optuna dan fitting final terpisah.


## Batas kesimpulan [Inti]

Jika seluruh pemeriksaan cocok, audit berhasil menelusuri pembagian asli. Artinya grup yang ditetapkan protokol tidak melintasi partisi; belum berarti train dan test bebas dari **semua kemiripan kimia**.

Test pernah dilihat pada pengembangan sebelumnya. Karena itu hasil tetap disebut **eksploratori**: temuan pada protokol ini, belum konfirmasi pada data eksternal yang sepenuhnya baru.

Lanjut ke **05_Model_dan_Bobot.ipynb** untuk melihat model dan bobot kelas.
